# MINI Cells — Experiment 005: 500K Consumer Language Model Bridge

This notebook compares three models on the same TinyStories token stream and exactly 500,000 supervised training tokens per model:

1. `textnca-s` — structural TextNCA control;
2. `minitextnca-s-plus` — RMSNorm + carry-biased GRU + stage auxiliary losses;
3. `transformer-s` — automatically parameter-matched Transformer baseline.

Checkpoints are evaluated at 125K, 250K and 500K tokens. The experiment is intentionally independent of PVM/JAM constraints.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path('/kaggle/working/mini-cells')
os.chdir('/kaggle/working')

if not (ROOT / '.git').exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ArcheLabs/mini-cells.git', str(ROOT),
    ], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=ROOT, check=True)
    subprocess.run(['git', 'switch', 'main'], cwd=ROOT, check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=ROOT, check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'], cwd=ROOT, check=True)
os.chdir(ROOT)
print('repo:', ROOT)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
print('cuda check:')
subprocess.run([sys.executable, '-c', 'import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', 'tests/research/01-foundations/test_language_bridge.py', '-q'], cwd=ROOT, check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/research/run_consumer_language_bridge.py'], cwd=ROOT, check=True)


In [ ]:
import json
from IPython.display import Image, Markdown, display

OUT = ROOT / 'results' / 'consumer-language-bridge-v1'
decision = json.loads((OUT / 'decision.json').read_text(encoding='utf-8'))
display(Markdown(f"## {decision['status']}: {decision['diagnosis']}"))
for name in [
    'consumer-readiness-summary.png',
    'ppl-scaling.png',
    'relative-gap.png',
    'learning-slope.png',
    'throughput.png',
]:
    display(Image(filename=str(OUT / name)))
display(Markdown((OUT / 'generation-progression.md').read_text(encoding='utf-8')))


In [ ]:
# Set this to True after reviewing the experiment outputs.
PUBLISH = False
if PUBLISH:
    subprocess.run([
        sys.executable, 'scripts/research/publish_experiment_results.py', '005', '--push'
    ], cwd=ROOT, check=True)
